## Stock Market Drift Detection using Yfinance

In [1]:
import numpy as np 
import pandas as pd 

In [2]:
import yfinance as yf
ticker = "^GSPC"  # S&P 500 index — reflects broad market regime shifts
df = yf.download(ticker, start="2005-01-01", end="2024-12-31")
df = df.reset_index()
df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]  # flatten if multiindex
print(df.shape)
df.head()

[*********************100%***********************]  1 of 1 completed

(5032, 6)


,Date,Close,High,Low,Open,Volume
0,2005-01-03,1202.079956,1217.800049,1200.319946,1211.920044,1510800000
1,2005-01-04,1188.050049,1205.839966,1185.390015,1202.079956,1721000000
2,2005-01-05,1183.739990,1192.729980,1183.719971,1188.050049,1738900000
3,2005-01-06,1187.890015,1191.630005,1183.270020,1183.739990,1569100000
4,2005-01-07,1186.189941,1192.199951,1182.160034,1187.890015,1477900000


In [3]:
print(df["Date"].min(), df["Date"].max())
print(df.isnull().sum())

2005-01-03 00:00:00 2024-12-30 00:00:00
Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64


In [4]:
df = df.sort_values("Date").reset_index(drop=True)

In [6]:
df["return_1d"] = df["Close"].pct_change()
df["ma_5"] = df["Close"].rolling(5).mean()
df["ma_20"] = df["Close"].rolling(20).mean()
df["ma_50"] = df["Close"].rolling(50).mean()
df["volatility_10d"] = df["return_1d"].rolling(10).std()
df["volume_change"] = df["Volume"].pct_change()

# RSI (14-day)
delta = df["Close"].diff()
gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)
avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()
rs = avg_gain / avg_loss
df["rsi_14"] = 100 - (100 / (1 + rs))

# MACD
ema_12 = df["Close"].ewm(span=12, adjust=False).mean()
ema_26 = df["Close"].ewm(span=26, adjust=False).mean()
df["macd"] = ema_12 - ema_26
df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()

df.head(10) 

,Date,Close,High,Low,Open,Volume,return_1d,ma_5,ma_20,ma_50,volatility_10d,volume_change,rsi_14,macd,macd_signal
0,2005-01-03,1202.079956,1217.800049,1200.319946,1211.920044,1510800000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000
1,2005-01-04,1188.050049,1205.839966,1185.390015,1202.079956,1721000000,-0.011671,NaN,NaN,NaN,NaN,0.139132,NaN,-1.119195,-0.223839
2,2005-01-05,1183.739990,1192.729980,1183.719971,1188.050049,1738900000,-0.003628,NaN,NaN,NaN,NaN,0.010401,NaN,-2.327125,-0.644496
3,2005-01-06,1187.890015,1191.630005,1183.270020,1183.739990,1569100000,0.003506,NaN,NaN,NaN,NaN,-0.097648,NaN,-2.915933,-1.098784
4,2005-01-07,1186.189941,1192.199951,1182.160034,1187.890015,1477900000,-0.001431,1189.589990,NaN,NaN,NaN,-0.058122,NaN,-3.479638,-1.574954
5,2005-01-10,1190.250000,1194.780029,1184.800049,1186.189941,1490400000,0.003423,1187.223999,NaN,NaN,NaN,0.008458,NaN,-3.557754,-1.971514
6,2005-01-11,1182.989990,1190.250000,1180.430054,1190.250000,1488800000,-0.006100,1186.211987,NaN,NaN,NaN,-0.001074,NaN,-4.157558,-2.408723
7,2005-01-12,1187.699951,1187.920044,1175.640015,1182.989990,1562100000,0.003981,1187.003979,NaN,NaN,NaN,0.049234,NaN,-4.204387,-2.767856
8,2005-01-13,1177.449951,1187.699951,1175.810059,1187.699951,1510300000,-0.008630,1184.915967,NaN,NaN,NaN,-0.033160,NaN,-5.010827,-3.216450
9,2005-01-14,1184.520020,1185.209961,1177.449951,1177.449951,1335400000,0.006005,1184.581982,NaN,NaN,NaN,-0.115805,NaN,-5.021555,-3.577471


In [7]:
df["target"] = (df["Close"].shift(-1) > df["Close"]).astype(int)

In [8]:
feature_cols = ["return_1d", "ma_5", "ma_20", "ma_50", "volatility_10d",
                 "volume_change", "rsi_14", "macd", "macd_signal"]

df_clean = df.dropna(subset=feature_cols + ["target"]).reset_index(drop=True)
print(df_clean.shape)
print(df_clean["target"].value_counts(normalize=True))

(4983, 16)
target
1    0.544451
0    0.455549
Name: proportion, dtype: float64


In [9]:
final_df = df_clean[feature_cols + ["target", "Date"]].copy()
final_df.to_csv("stock_market_cleaned.csv", index=False)
final_df.head()

,return_1d,ma_5,ma_20,ma_50,volatility_10d,volume_change,rsi_14,macd,macd_signal,target,Date
0,-0.007524,1204.183984,1206.583496,1194.796199,0.006368,0.052942,53.610355,2.807775,4.555855,0,2005-03-15
1,-0.008082,1200.395972,1205.480994,1194.515999,0.006732,0.092274,43.717002,1.103743,3.865433,1,2005-03-16
2,0.001801,1196.587964,1204.474493,1194.559197,0.006800,-0.043105,37.908603,-0.073189,3.077708,0,2005-03-17
3,-0.000470,1194.501978,1203.919495,1194.677397,0.005578,0.481968,41.312802,-1.039121,2.254342,0,2005-03-18
4,-0.004934,1189.891992,1203.028998,1194.595198,0.005286,-0.223911,33.219955,-2.252324,1.353009,0,2005-03-21
